# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before testing any signal, I look at the raw distributions of my key fields to
check for heavy tails, extreme outliers, or unexpected zero-inflation that could
distort later averages.

In [ ]:
dist_check = con.sql(f"""
    SELECT
        MIN(gsc_impressions) AS min_impr,
        approx_quantile(gsc_impressions, 0.5) AS median_impr,
        approx_quantile(gsc_impressions, 0.95) AS p95_impr,
        MAX(gsc_impressions) AS max_impr,
        MIN(gsc_clicks) AS min_clicks,
        approx_quantile(gsc_clicks, 0.5) AS median_clicks,
        approx_quantile(gsc_clicks, 0.95) AS p95_clicks,
        MAX(gsc_clicks) AS max_clicks,
        MIN(gsc_avg_position) AS min_pos,
        approx_quantile(gsc_avg_position, 0.5) AS median_pos,
        MAX(gsc_avg_position) AS max_pos
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
dist_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_impr,median_impr,p95_impr,max_impr,min_clicks,median_clicks,p95_clicks,max_clicks,min_pos,median_pos,max_pos
0,1,16,337,40084,0,0,1,274,0.0,7.482218,498.0


**Observation:** Impressions are heavily right-skewed — median is just 16, but the
95th percentile jumps to 337 and the max reaches 40,084. Clicks show the same
pattern even more sharply: the median is 0 (most content gets zero clicks in a
month), while a handful of pages reach up to 274 clicks. Position ranges from 0
to 498 — some content ranks extremely poorly. This heavy tail means averages
computed later (like avg_clicks per bucket) are pulled upward by a small number
of very high-traffic pages; I report bucket counts (n) alongside every average
to keep this honest.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three signal tests, each with a mini-test and a verdict (CONFIRMED / OPPOSITE /
MIXED / FALSE).

In [ ]:
sig1 = con.sql(f"""
    WITH content_perf AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) AS impressions, SUM(f.gsc_clicks) AS clicks
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
        WHERE f.gsc_data_available IS TRUE
        GROUP BY f.client_hash_id, f.content_hash_id
    )
    SELECT
        CASE WHEN c.content_updated_date > DATE '2026-03-31' OR c.content_updated_date IS NULL
             THEN 'stale (not updated by March)' ELSE 'fresh (updated by March)' END AS bucket,
        COUNT(*) AS n, AVG(cp.clicks) AS avg_clicks, AVG(cp.impressions) AS avg_impressions
    FROM content_perf cp
    JOIN read_parquet('{BASE}/dim_content.parquet') c
      ON cp.content_hash_id = c.content_hash_id AND cp.client_hash_id = c.client_hash_id
    WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
    GROUP BY bucket
""").df()
sig1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n,avg_clicks,avg_impressions
0,fresh (updated by March),27801,2.807345,1239.622963
1,stale (not updated by March),148767,4.998622,1654.639524


**Signal Test #1 — Staleness — Verdict: OPPOSITE**
Stale content (n=148,767) averaged more clicks and impressions than fresh content
(n=27,801) — the reverse of the assumption that stale content underperforms.
Likely explanation: unupdated content may be older, established, high-traffic
pages that simply don't need updates yet.

In [ ]:
sig2 = con.sql(f"""
    WITH content_perf AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        CASE WHEN avg_position <= 10 THEN 'good_position (top 10)' ELSE 'poor_position (below 10)' END AS bucket,
        COUNT(*) AS n, AVG(clicks * 1.0 / impressions) AS avg_ctr
    FROM content_perf
    GROUP BY bucket
""").df()
sig2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n,avg_ctr
0,good_position (top 10),55895,0.003294
1,poor_position (below 10),45546,0.001784


**Signal Test #2 — CTR vs Position — Verdict: CONFIRMED**
Top-10 position content (n=55,895) averaged roughly double the CTR of below-10
content (n=45,546) — in the expected direction. Position and CTR are related as
assumed.

In [ ]:
sig3 = con.sql(f"""
    WITH content_perf AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        CASE WHEN c.search_volume >= 1000 THEN 'high_volume (>=1000)' ELSE 'low_volume (<1000)' END AS bucket,
        COUNT(*) AS n, AVG(cp.clicks) AS avg_clicks
    FROM content_perf cp
    JOIN read_parquet('{BASE}/dim_content.parquet') c
      ON cp.content_hash_id = c.content_hash_id AND cp.client_hash_id = c.client_hash_id
    WHERE c.search_volume IS NOT NULL
    GROUP BY bucket
""").df()
sig3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n,avg_clicks
0,low_volume (<1000),157380,5.093792
1,high_volume (>=1000),3426,3.305020


**Signal Test #3 — Search volume vs actual clicks — Verdict: OPPOSITE**
High-volume keywords (search_volume >= 1000, n=3,426) averaged only 3.31 clicks,
while low-volume keywords (n=157,380) averaged 5.09 clicks — the reverse of the
"quick win" assumption that high search volume predicts high traffic capture.
Likely explanation: high-volume keywords face more competition, so any single
piece of content captures a smaller share of that larger pie, while low-volume
keywords have less competition and let a single ranking page capture more of
the (smaller) total demand.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

This test picks the signal behind FlyRank's real **refresh flag**: the assumption
that stale (unupdated) content underperforms and needs refreshing.

In [ ]:
sig1  # same result as Section 2, Test #1


,bucket,n,avg_clicks,avg_impressions
0,fresh (updated by March),27801,2.807345,1239.622963
1,stale (not updated by March),148767,4.998622,1654.639524


**Does the data support the refresh flag's assumption? NO.**
The refresh flag assumes staleness signals decline, but the March 2026 data shows
the opposite direction — stale content outperforms fresh content on both clicks
and impressions. This doesn't mean the refresh flag is useless (staleness may
still matter for specific declining pages), but it means "stale = automatically
worth reviewing" is not supported as a blanket rule on this slice.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should not treat "hasn't been updated recently" as a standalone
red flag — stale content in this data is often the strongest performer, likely
older, proven content. Similarly, "high search volume" alone doesn't guarantee
more traffic — competition eats into that potential. The one signal that holds
up is CTR-vs-position: pages ranking well but getting unusually low clicks are a
real, actionable opportunity, and should be prioritized over staleness or raw
search volume when deciding what to review first.

In [ ]:
print("Summary of verdicts:")
print("Signal 1 (staleness):", "OPPOSITE")
print("Signal 2 (CTR vs position):", "CONFIRMED")
print("Signal 3 (search volume vs clicks):", "OPPOSITE")
print("Flag-linked test (refresh flag / staleness): NOT supported by this data slice")

Summary of verdicts:
Signal 1 (staleness): OPPOSITE
Signal 2 (CTR vs position): CONFIRMED
Signal 3 (search volume vs clicks): OPPOSITE
Flag-linked test (refresh flag / staleness): NOT supported by this data slice


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.